### RAG System: QA with PDF

QA with PDF RAG means using a Question Answering (QA) system powered by Retrieval-Augmented Generation (RAG) on PDF documents.

**Workflow**

1. 	Load PDF → Extract text.
2. 	Chunk text → Split into smaller sections.
3. 	Embed chunks → Store embeddings in a vector database (e.g., FAISS, Pinecone, Chroma).
4. 	User asks a question → Convert question into an embedding.
5. 	Retrieve relevant chunks → Find the closest matches in the vector DB.
6. 	Generate answer → Pass retrieved text + question into the LLM to produce a coherent answer.

In [52]:
import os

# from dotenv import load_dotenv
# load_dotenv()
# hf_api_key = os.getenv("HUGGINGFACE_API_KEY")
# groq_api_key = os.getenv("GROQ_API_KEY")

from google.colab import userdata
hf_api_key = userdata.get('hf_token')
groq_api_key = userdata.get('GROQ_API_KEY')




In [8]:
!pip install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [13]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.5/334.5 kB 7.9 MB/s eta 0:00:00


In [25]:
!pip install langchain-huggingface

In [33]:
# For CPU use following
!pip install faiss-cpu

# For GPU use following
# !pip install faiss-gpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 26.2 MB/s eta 0:00:00


In [41]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.0 MB/s eta 0:00:00


In [53]:
!pip show langchain-community pypdf langchain-huggingface faiss-cpu langchain-groq langchain-text-splitters


Name: langchain-community
Version: 0.4.1
Summary: Community contributed LangChain integrations.
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: aiohttp, dataclasses-json, httpx-sse, langchain-classic, langchain-core, langsmith, numpy, pydantic-settings, PyYAML, requests, SQLAlchemy, tenacity
Required-by: 
---
Name: pypdf
Version: 6.10.0
Summary: A pure-python PDF library capable of splitting, merging, cropping, and transforming PDF files
Home-page: 
Author: 
Author-email: Mathieu Fenniak <biziqe@mathieu.fenniak.net>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: 
Required-by: 
---
Name: langchain-huggingface
Version: 1.2.1
Summary: An integration package connecting Hugging Face and LangChain.
Home-page: https://docs.langchain.com/oss/python/integrations/providers/huggingface
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface-hub, langchain-core, 

In [54]:
import importlib.metadata as md

packages = [
    "langchain-community",
    "pypdf",
    "langchain-huggingface",
    "faiss-cpu",
    "langchain-groq",
    "langchain-text-splitters"
]

for pkg in packages:
    try:
        print(f"{pkg}: {md.version(pkg)}")
    except:
        print(f"{pkg}: Not installed")


langchain-community: 0.4.1
pypdf: 6.10.0
langchain-huggingface: 1.2.1
faiss-cpu: 1.13.2
langchain-groq: 1.1.2
langchain-text-splitters: 1.1.1


In [27]:
# # To read this file I will use the langchain document load that will load this document in standard format - langchain document - page and metadata

from langchain_community.document_loaders import PyPDFLoader
doc_path = "https://raw.githubusercontent.com/ash322ash422/data/main/document-loxford-company.pdf"
loader = PyPDFLoader(doc_path)
loader

In [28]:
# load the doc


docs = loader.load()
docs

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-25T08:17:02+05:30', 'author': 'Ashwani Kumar', 'moddate': '2026-03-25T08:17:02+05:30', 'source': 'https://raw.githubusercontent.com/ash322ash422/data/main/document-loxford-company.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='Loxford Technologies – Company Overview \nLoxford Technologies is a global technology solutions provider specializing in \nartificial intelligence, data analytics, and enterprise automation. Founded in 2012, \nthe company has grown rapidly into a mid-sized enterprise serving clients across \nNorth America, Europe, and Asia. \nLoxford Technologies was founded by Daniel Reeves, a former data scientist, and \nAnika Lomax, a software engineer with a background in distributed systems. The \nfounders envisioned a company that could bridge the gap between cutting-edge \nAI research and real-world business applications. \n• CEO: Daniel

In [29]:
# To read this
import pprint

print("Metadata:")
pprint.pp(docs[0].metadata)
print("-"*25)
print("Page Content:")
pprint.pp(docs[0].page_content)

Metadata:
{'producer': 'Microsoft® Word 2021',
 'creator': 'Microsoft® Word 2021',
 'creationdate': '2026-03-25T08:17:02+05:30',
 'author': 'Ashwani Kumar',
 'moddate': '2026-03-25T08:17:02+05:30',
 'source': 'https://raw.githubusercontent.com/ash322ash422/data/main/document-loxford-company.pdf',
 'total_pages': 3,
 'page': 0,
 'page_label': '1'}
-------------------------
Page Content:
('Loxford Technologies – Company Overview \n'
 'Loxford Technologies is a global technology solutions provider specializing '
 'in \n'
 'artificial intelligence, data analytics, and enterprise automation. Founded '
 'in 2012, \n'
 'the company has grown rapidly into a mid-sized enterprise serving clients '
 'across \n'
 'North America, Europe, and Asia. \n'
 'Loxford Technologies was founded by Daniel Reeves, a former data scientist, '
 'and \n'
 'Anika Lomax, a software engineer with a background in distributed systems. '
 'The \n'
 'founders envisioned a company that could bridge the gap between '
 'cu

In [30]:
# Split the documents in manageable chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap = 50
)

doc_chunks = splitter.split_documents(docs)

In [19]:
len(doc_chunks)

12

In [18]:
doc_chunks

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-25T08:17:02+05:30', 'author': 'Ashwani Kumar', 'moddate': '2026-03-25T08:17:02+05:30', 'source': 'https://raw.githubusercontent.com/ash322ash422/data/main/document-loxford-company.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='Loxford Technologies – Company Overview \nLoxford Technologies is a global technology solutions provider specializing in \nartificial intelligence, data analytics, and enterprise automation. Founded in 2012, \nthe company has grown rapidly into a mid-sized enterprise serving clients across'),
 Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-25T08:17:02+05:30', 'author': 'Ashwani Kumar', 'moddate': '2026-03-25T08:17:02+05:30', 'source': 'https://raw.githubusercontent.com/ash322ash422/data/main/document-loxford-company.pdf', 'total_pages': 3, 'page': 0, 'page_labe

In [31]:
for doc in doc_chunks:
    print(doc.page_content)
    print("-"*25)

Loxford Technologies – Company Overview 
Loxford Technologies is a global technology solutions provider specializing in 
artificial intelligence, data analytics, and enterprise automation. Founded in 2012, 
the company has grown rapidly into a mid-sized enterprise serving clients across
-------------------------
North America, Europe, and Asia. 
Loxford Technologies was founded by Daniel Reeves, a former data scientist, and 
Anika Lomax, a software engineer with a background in distributed systems. The 
founders envisioned a company that could bridge the gap between cutting-edge
-------------------------
AI research and real-world business applications. 
• CEO: Daniel Reeves 
• CTO: Anika Lomax 
The company is headquartered in Austin, Texas, USA, with additional offices in: 
• London, UK 
• Bengaluru, India 
• Berlin, Germany
-------------------------
• Bengaluru, India 
• Berlin, Germany 
As of 2024, Loxford Technologies reported an estimated annual revenue of $180 
million, with a ye

In [22]:
# check somethings

doc_chunks[5].page_content

'• RAG-Based Knowledge Systems \nEnterprise-grade Retrieval-Augmented Generation systems for internal \nknowledge management.'

In [23]:
doc_chunks[5].metadata

{'producer': 'Microsoft® Word 2021',
 'creator': 'Microsoft® Word 2021',
 'creationdate': '2026-03-25T08:17:02+05:30',
 'author': 'Ashwani Kumar',
 'moddate': '2026-03-25T08:17:02+05:30',
 'source': 'https://raw.githubusercontent.com/ash322ash422/data/main/document-loxford-company.pdf',
 'total_pages': 3,
 'page': 0,
 'page_label': '1'}

In [44]:
# Let's do the embedding

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectordb = FAISS.from_documents(doc_chunks, embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


- Here I have used HuggingFaceEmbeddings from langchain to load llm for creating embeddings.
- This loads the **MiniLM sentence transformer from Hugging Face**.
- It's lightweight, fast, and very accurate for semantic search.
- Converts each text chunk into a 768-dimensional vector that captures its meaning.
- Converts all the doc_chunks into embeddings using the model.
- Stores them in a FAISS index, which supports fast nearest neighbor search.
- Returns a vectordb object that you can query later.
- Many VectorDB - FAISS, CromaDB, pinecone etc

In [35]:
print(f"Stored {vectordb.index.ntotal} chunks in the FAISS vector store.")

Stored 12 chunks in the FAISS vector store.


In [38]:
# Perform the similarity search

result = vectordb.similarity_search("Who founded of Loxford company?")

result

[Document(id='60b47b42-d9f9-4b93-9580-412ea50aa57c', metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-25T08:17:02+05:30', 'author': 'Ashwani Kumar', 'moddate': '2026-03-25T08:17:02+05:30', 'source': 'https://raw.githubusercontent.com/ash322ash422/data/main/document-loxford-company.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='North America, Europe, and Asia. \nLoxford Technologies was founded by Daniel Reeves, a former data scientist, and \nAnika Lomax, a software engineer with a background in distributed systems. The \nfounders envisioned a company that could bridge the gap between cutting-edge'),
 Document(id='9e7078b1-8f30-4a1a-8022-bda73308a770', metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-25T08:17:02+05:30', 'author': 'Ashwani Kumar', 'moddate': '2026-03-25T08:17:02+05:30', 'source': 'https://raw.githubusercontent.com/ash322ash422/data/main/

In [50]:
len(result)

4

In [51]:
import os
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA

llm = ChatGroq(model="qwen/qwen3-32b", api_key=groq_api_key)

qa = RetrievalQA.from_chain_type(llm, chain_type = "stuff", retriever=vectordb.as_retriever())

In [49]:
question = "Who founded Loxford company ?"

answer = qa.invoke(question)

print(answer)
print(answer["result"])

{'query': 'Who founded Loxford company ?', 'result': '<think>\nOkay, the user is asking who founded Loxford Technologies. Let me check the provided context.\n\nLooking at the first paragraph under "Loxford Technologies – Company Overview," it says the company was founded by Daniel Reeves and Anika Lomax. Daniel is a former data scientist, and Anika is a software engineer with a background in distributed systems. The founders\' names are clearly mentioned there. \n\nI need to make sure there\'s no other information conflicting or additional names. The rest of the context talks about the company\'s specialization, locations, revenue, and culture, but no other founders are mentioned. \n\nSo the answer should include both Daniel Reeves and Anika Lomax, their roles, and maybe their backgrounds as provided. The user didn\'t specify needing more details, but including their backgrounds adds value. \n\nI should structure the answer clearly, stating both names and their respective backgrounds. 